In [1]:
import os
import torch
import torchvision

# 1. Versions (useful if something breaks and you need to search the error)
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)

# 2. Is the GPU visible to PyTorch?
gpu_ok = torch.cuda.is_available()
print("GPU available:", gpu_ok)
if gpu_ok:
    print("GPU name:", torch.cuda.get_device_name(0))

# 3. One variable that decides where all computation happens
device = torch.device("cuda" if gpu_ok else "cpu")
print("Using device:", device)

# 4. Show the dataset structure, only 3 levels deep, with file counts
#    (never print every file: there are 120,000 of them)
root = "/kaggle/input"
for dirpath, dirnames, filenames in os.walk(root):
    depth = dirpath[len(root):].count(os.sep)
    if depth > 3:
        continue
    indent = "  " * depth
    print(f"{indent}{os.path.basename(dirpath) or root}/  ({len(filenames)} files)")

PyTorch: 2.11.0+cu128 | torchvision: 0.26.0+cu128
GPU available: True
GPU name: Tesla T4
Using device: cuda
input/  (0 files)
  datasets/  (0 files)
    birdy654/  (0 files)
      cifake-real-and-ai-generated-synthetic-images/  (0 files)


In [2]:
DATA_DIR = "/kaggle/input/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images"

for split in sorted(os.listdir(DATA_DIR)):
    split_path = os.path.join(DATA_DIR, split)
    if os.path.isdir(split_path):
        for cls in sorted(os.listdir(split_path)):
            n = len(os.listdir(os.path.join(split_path, cls)))
            print(f"{split}/{cls}: {n} images")
            

test/FAKE: 10000 images
test/REAL: 10000 images
train/FAKE: 50000 images
train/REAL: 50000 images


In [3]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

IMG_SIZE = 224
BATCH_SIZE = 64

# ImageNet's average colors and spread (what EfficientNet was trained with)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# The "preparation steps" applied to every image, in order
tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),  # 32x32 -> 224x224
    transforms.ToTensor(),                     # picture -> grid of numbers (0 to 1)
    transforms.Normalize(MEAN, STD),           # match the model's "units"
])

# Folder names become labels automatically
full_train = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=tf)
test_ds    = datasets.ImageFolder(os.path.join(DATA_DIR, "test"),  transform=tf)
print("Label mapping:", full_train.class_to_idx)   # remember this for the API!

# Split 100K training images into 90K train + 10K validation, same split every run
g = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(full_train, [90_000, 10_000], generator=g)
print(f"Train: {len(train_ds)} | Validation: {len(val_ds)} | Test: {len(test_ds)}")

# Batch feeders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

# Sanity check: grab one batch and look at it
images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)    # expect [64, 3, 224, 224]
print("Labels in this batch: FAKE =", (labels == 0).sum().item(),
      "| REAL =", (labels == 1).sum().item())

Label mapping: {'FAKE': 0, 'REAL': 1}
Train: 90000 | Validation: 10000 | Test: 20000
Batch shape: torch.Size([64, 3, 224, 224])
Labels in this batch: FAKE = 37 | REAL = 27


In [4]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# 1. Download EfficientNet-B0 with its ImageNet "experience"
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
print("Original last layer:", model.classifier)

# 2. Replace the last layer: 1,000 object classes -> 2 classes (FAKE, REAL)
in_features = model.classifier[1].in_features   # 1280 numbers come into this layer
model.classifier[1] = nn.Linear(in_features, 2)
print("New last layer:     ", model.classifier)

# 3. Move the model onto the GPU
model = model.to(device)

# 4. The grading system
criterion = nn.CrossEntropyLoss()

# 5. The improvement method, with small careful steps
LEARNING_RATE = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# Model size
n_params = sum(p.numel() for p in model.parameters())
print(f"Trainable weights: {n_params:,}")

# Sanity check: push the one batch we inspected through the (untrained) model
with torch.no_grad():
    out = model(images.to(device))
print("Output shape:", out.shape)    # expect [64, 2]: two scores per image

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 143MB/s] 


Original last layer: Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)
New last layer:      Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=2, bias=True)
)
Trainable weights: 4,010,110
Output shape: torch.Size([64, 2])


In [5]:
import time, json
from torch.amp import autocast, GradScaler

EPOCHS = 3
SAVE_PATH = "/kaggle/working/verify_effnetb0_best.pth"
scaler = GradScaler("cuda")   # helper for the mixed-precision speed trick

def evaluate(loader):
    """Score the model on a dataset without learning from it."""
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with autocast("cuda"):
                out = model(x)
                loss = criterion(out, y)
            preds = out.argmax(1)
            loss_sum += loss.item() * y.size(0)
            correct += (preds == y).sum().item()
            total += y.size(0)
            all_preds.append(preds.cpu()); all_labels.append(y.cpu())
    return loss_sum / total, correct / total, torch.cat(all_preds), torch.cat(all_labels)

best_val_acc, history = 0.0, []

for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    run_loss, run_correct, seen = 0.0, 0, 0

    for i, (x, y) in enumerate(train_loader):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        optimizer.zero_grad()                 # 5. reset last batch's corrections
        with autocast("cuda"):
            out = model(x)                    # 1. guess
            loss = criterion(out, y)          # 2. grade
        scaler.scale(loss).backward()         # 3. work out corrections
        scaler.step(optimizer)                # 4. apply corrections
        scaler.update()

        run_loss += loss.item() * y.size(0)
        run_correct += (out.argmax(1) == y).sum().item()
        seen += y.size(0)
        if (i + 1) % 200 == 0:
            print(f"  epoch {epoch+1} | batch {i+1}/{len(train_loader)} | "
                  f"loss {run_loss/seen:.4f} | acc {run_correct/seen:.4f}")

    train_loss, train_acc = run_loss / seen, run_correct / seen
    val_loss, val_acc, _, _ = evaluate(val_loader)
    mins = (time.time() - t0) / 60
    print(f"EPOCH {epoch+1}: train acc {train_acc:.4f} | val acc {val_acc:.4f} | "
          f"val loss {val_loss:.4f} | {mins:.1f} min")
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc,
                    "val_loss": val_loss, "val_acc": val_acc, "minutes": mins})

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  -> new best model saved ({val_acc:.4f})")

  epoch 1 | batch 200/1407 | loss 0.2661 | acc 0.8898
  epoch 1 | batch 400/1407 | loss 0.2029 | acc 0.9179
  epoch 1 | batch 600/1407 | loss 0.1726 | acc 0.9311
  epoch 1 | batch 800/1407 | loss 0.1556 | acc 0.9381
  epoch 1 | batch 1000/1407 | loss 0.1436 | acc 0.9436
  epoch 1 | batch 1200/1407 | loss 0.1348 | acc 0.9472
  epoch 1 | batch 1400/1407 | loss 0.1272 | acc 0.9505
EPOCH 1: train acc 0.9505 | val acc 0.9780 | val loss 0.0606 | 10.4 min
  -> new best model saved (0.9780)
  epoch 2 | batch 200/1407 | loss 0.0588 | acc 0.9787
  epoch 2 | batch 400/1407 | loss 0.0563 | acc 0.9794
  epoch 2 | batch 600/1407 | loss 0.0556 | acc 0.9794
  epoch 2 | batch 800/1407 | loss 0.0551 | acc 0.9797
  epoch 2 | batch 1000/1407 | loss 0.0547 | acc 0.9797
  epoch 2 | batch 1200/1407 | loss 0.0549 | acc 0.9796
  epoch 2 | batch 1400/1407 | loss 0.0550 | acc 0.9797
EPOCH 2: train acc 0.9797 | val acc 0.9825 | val loss 0.0507 | 17.5 min
  -> new best model saved (0.9825)
  epoch 3 | batch 200/14

In [6]:
# Load the best version, then take the final exam ONCE
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
test_loss, test_acc, preds, labels = evaluate(test_loader)

# Remember: FAKE = 0, REAL = 1
fake_caught   = ((preds == 0) & (labels == 0)).sum().item()
fake_missed   = ((preds == 1) & (labels == 0)).sum().item()
real_correct  = ((preds == 1) & (labels == 1)).sum().item()
real_flagged  = ((preds == 0) & (labels == 1)).sum().item()

false_positive_rate = real_flagged / (real_flagged + real_correct)  # real photos called AI
miss_rate           = fake_missed / (fake_missed + fake_caught)      # AI images called real

print(f"TEST ACCURACY: {test_acc:.4f}")
print(f"AI images caught: {fake_caught} | AI images missed: {fake_missed}")
print(f"Real images correct: {real_correct} | Real images wrongly flagged: {real_flagged}")
print(f"False positive rate (real flagged as AI): {false_positive_rate:.4f}")
print(f"Miss rate (AI passed as real): {miss_rate:.4f}")

# Save everything the API will need, plus your numbers
results = {
    "test_accuracy": test_acc, "false_positive_rate": false_positive_rate,
    "miss_rate": miss_rate, "best_val_acc": best_val_acc, "history": history,
    "class_to_idx": full_train.class_to_idx, "img_size": IMG_SIZE,
    "mean": MEAN, "std": STD, "learning_rate": LEARNING_RATE, "epochs": EPOCHS,
}
with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved results.json")

TEST ACCURACY: 0.9790
AI images caught: 9729 | AI images missed: 271
Real images correct: 9852 | Real images wrongly flagged: 148
False positive rate (real flagged as AI): 0.0148
Miss rate (AI passed as real): 0.0271
Saved results.json
